In [1]:
import pandas as pd
import openai
import os  
from tqdm import tqdm
import re

In [2]:
question_list= [
    # plan CT
"""
The note has a "Plan" or "Assesment/Plan" section, which is towards the end of the note.
Does it mention the patient require a CT scan in the plan section? 

Let's think step by step.\n"
    "[REASONING]"
    "Thus, the answer is: [ANSWER: Yes | No]

I want you to output the ANSWER in a json format

{plan_CT: ANSWER}

""",
 # CTA
"""
The note has a "Plan" or "Assesment/Plan" section, which is towards the end of the note.
Does it mention the patient require a Computed Tomography Angiography (CTA) scan in the plan section? 

Let's think step by step.\n"
    "[REASONING]"
    "Thus, the answer is: [ANSWER: Yes | No]

I want you to output the ANSWER in a json format

{plan_CTA: ANSWER}

""",
 # MRI
"""
The note has a "Plan" or "Assesment/Plan" section, which is towards the end of the note.
Does it mention the patient require a MRI scan in the plan section? 

Let's think step by step.\n"
    "[REASONING]"
    "Thus, the answer is: [ANSWER: Yes | No]

I want you to output the ANSWER in a json format

{plan_MRI: ANSWER}

""",
 # MRA
"""
The note has a "Plan" or "Assesment/Plan" section, which is towards the end of the note.
Does it mention the patient require a MRA scan in the plan section? 

Let's think step by step.\n"
    "[REASONING]"
    "Thus, the answer is: [ANSWER: Yes | No]

I want you to output the ANSWER in a json format

{plan_MRA: ANSWER}

""",
# Pediatric Stroke Outcome Measure - 
"""
What is the total score of the Pediatric Stroke Outcome Measure? 

Let's think step by step.\n"
    "[REASONING]"
    "Thus, the answer is:[ANSWER]

I want you to output the ANSWER in a json format

{pediatric_stroke_outcome_measure_score: ANSWER}
""",
    
]

# 1st round

In [3]:
def convert_narrative(filename):
    df = pd.read_excel(filename)

    # Creating an empty list to store generated sentences
    sentences = []


#     # Function to format sentences
#     def format_sentences(sentences):
#         formatted_sentences = []
#         for i, sentence in enumerate(sentences, start=1):
#             formatted_sentence = f"[Sentence ID {i}] {sentence.strip()}"
#             formatted_sentences.append(formatted_sentence)
#         return formatted_sentences, i

    # Processing each row in the dataframe
    sentence_id = 1
    for index, row in df.iterrows():
        # Checking for the existence of data in the 'line' column
        if pd.notna(row['line']):
            # Generating the descriptive sentence based on the available information

#             if index == 0: # narrative 
#                 sentences_list = row['value'].split('.')
#                 formatted_sentences, i = format_sentences(sentences_list)
#                 sentences.extend(formatted_sentences)
#                 sentence_id = i
#             else:
            
            sentence = f"[Sentence ID {sentence_id}] {row['value']}"
            sentences.append(sentence)
            sentence_id += 1

    result = '. \n '.join(sentences)
    return result

In [31]:
openai.api_type = "azure"
openai.api_version = "2023-05-15"
openai.api_base = "https://ontology-kz.openai.azure.com/"  # Your Azure OpenAI resource's endpoint value.
openai.api_key = ""


output_directory = './output_result_score_and_imaging/'

# Check if the directory exists, if not, create it
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
    

for filename in tqdm(os.listdir('./data')):

    save_filename = 'result_score_and_imaging_' + filename

    if os.path.exists(output_directory + save_filename):
        print(f'The file {save_filename} already exists.')
        continue

    with open('./data/' + filename, 'r') as f:
        notes = f.read()

    for question in question_list:
        
        response = openai.ChatCompletion.create(
        deployment_id = "mCodegpt_gpt_35_16k", # The deployment name you chose when you deployed the GPT-3.5-Turbo or GPT-4 model.
        messages = [
            {"role": "system", "content": "Assistant is a large language model trained by OpenAI."},
            {"role": "user", "content": notes + question}
        ]
        )

        print(response['usage'])

        result = response['choices'][0]['message']['content']

        with open(output_directory + save_filename, 'a') as f:
            f.write(result + '\n')



  0%|                                                                                                                                          | 0/50 [00:00<?, ?it/s]

The file result_score_and_imaging_30073.txt already exists.
The file result_score_and_imaging_30075.txt already exists.
The file result_score_and_imaging_30075_2.txt already exists.
The file result_score_and_imaging_30078.txt already exists.
The file result_score_and_imaging_30077.txt already exists.
The file result_score_and_imaging_30078_2.txt already exists.
The file result_score_and_imaging_30083_2.txt already exists.
The file result_score_and_imaging_30079.txt already exists.
The file result_score_and_imaging_30084.txt already exists.
The file result_score_and_imaging_30084_2.txt already exists.
The file result_score_and_imaging_30083.txt already exists.
The file result_score_and_imaging_30086.txt already exists.
The file result_score_and_imaging_30087.txt already exists.
The file result_score_and_imaging_30088.txt already exists.
The file result_score_and_imaging_30087_2.txt already exists.
The file result_score_and_imaging_30089.txt already exists.
The file result_score_and_imag

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 29.59it/s]

{
  "prompt_tokens": 1915,
  "completion_tokens": 29,
  "total_tokens": 1944
}


# Step 2: judge severity score:

Let's do this;  define a score of 0 as no deficit. a score of 0.5 as mild deficit, and a score of 1-2 as moderate. 2.5 or higher is severe. If GPT is able to figure out that, we could probably get the agreement much higher.# 2nd round: judge if 

In [ ]:
def determine_severity(score):
    if score == 0:
        return "No deficit"
    elif 0 < score <= 0.5:
        return "Mild deficit"
    elif 0.5 < score <= 2:
        return "Moderate deficit"
    elif score >= 2.5:
        return "Severe deficit"
    else:
        return "Invalid score"

for filename in tqdm(os.listdir(output_directory)):
    if not filename.endswith('txt'): continue

    with open(output_directory + filename, 'r') as f:
        previous_result = f.read()

        current_result = ""
        
        pattern = r'"pediatric_stroke_outcome_measure_score": (\d+)'

        match = re.search(pattern, previous_result)

        if match:
            pediatric_stroke_outcome_measure_score = int(match.group(1))
        else:
            pediatric_stroke_outcome_measure_score = 0
#         print(pediatric_stroke_outcome_measure_score)
#         input()
        severity = determine_severity(pediatric_stroke_outcome_measure_score)
    with open(output_directory + filename, 'a') as f:
        f.write('\n Severity is: ' + severity)
        

            

# step 4: zip the output

In [33]:
!zip -r output_result_score_and_imaging.zip output_result_score_and_imaging

  adding: output_result_score_and_imaging/ (stored 0%)
  adding: output_result_score_and_imaging/.ipynb_checkpoints/ (stored 0%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30073.txt (deflated 36%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30075.txt (deflated 42%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30075_2.txt (deflated 36%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30078.txt (deflated 37%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30077.txt (deflated 35%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30078_2.txt (deflated 38%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30083_2.txt (deflated 45%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30079.txt (deflated 34%)
  adding: output_result_score_and_imaging/result_score_and_imaging_30084.txt (deflated 27%)
  adding: output_result_score_and_ima